# Exploration of target variable

In [ ]:
%reload_ext autoreload

%autoreload 2

%reload_ext dotenv
%dotenv

In [ ]:
import dask
import dask.array
from dask.distributed import Client
import gc
import matplotlib
import matplotlib.pyplot as plt
import numpy as np
import xarray as xr

from mlde_analysis import create_map_fig, plot_map, cp_model_rotated_pole, osgb_crs, platecarree
from mlde_analysis.furflex_data import open_dataset_predictors_split
from mlde_analysis.display import pretty_table
from mlde_analysis.distribution import plot_freq_density, xr_hist

In [ ]:
client = Client()
client

In [ ]:
var = "psl"
dataset = "v3_engwales_ccpm-5km_100x12em_1hr_pr"
split = "val"
derived_variables_config = {}

In [ ]:
ds = open_dataset_predictors_split(dataset, split)
ds

In [ ]:
da = ds[var]

In [ ]:
stats = xr.merge([
    np.isnan(da).sum(...).rename(f"NaN Count"),
    da.mean().rename("mean"), 
    da.std().rename("std"),
    da.min().rename("min"), 
    da.quantile(0.25).drop_vars("quantile").rename("25%"),
    da.quantile(0.5).drop_vars("quantile").rename("50%"),
    da.quantile(0.75).drop_vars("quantile").rename("75%"),
    da.quantile(0.99).drop_vars("quantile").rename("99%"),
    da.quantile(0.999).drop_vars("quantile").rename("99.9%"),
    da.quantile(0.9999).drop_vars("quantile").rename("99.99%"),
    da.quantile(0.99999).drop_vars("quantile").rename("99.999%"),
    da.quantile(0.999999).drop_vars("quantile").rename("99.9999999%"), #~10 pixels higher in 12 em, 15 year validation dataset on a 14x14 60km grid
    da.max().rename("max"),
])
_ = pretty_table(stats.expand_dims(variable=[var]), round=3)

In [ ]:
bins=50
# bins = np.histogram_bin_edges([], bins=150, range=(-3, 3))

da.plot(label=var, density=True, bins=bins,)
plt.legend()
plt.yscale("log")
plt.show()

In [ ]:
def plot_hists(hists):
    for label, hist in hists.items():
        # hist.plot(label=label)
        plt.stairs(hist[0], hist[1], label=label)
    plt.legend()
    plt.yscale("log")
    plt.show()

def hist_compute(da):
    bins = np.histogram_bin_edges([], bins=200, range=(-3, 1))
    mean = da.mean().compute()
    std = da.std().compute()
    lim = max(np.abs((da.min()-mean)/std), np.abs((da.max()-mean)/std)).compute().item()
    bins = np.histogram_bin_edges([], bins=200, range=(-1.1*lim, 1.1*lim))
    
    return xr_hist((da-mean)/std, bins=bins)

plot_hists({
    "raw": hist_compute(da),
})

In [ ]:
time_mean = da.mean(dim=["ensemble_member", "time"])
time_std = da.std(dim=["ensemble_member", "time"])
time_q0001 = da.quantile(0.0001, dim=["ensemble_member", "time"]) # roughly equiv to 6 lower values in 12 em, 15 year dataset 
time_q001 = da.quantile(0.001, dim=["ensemble_member", "time"]) # rougly equiv to once every 3 years (at each grid box)
time_q999 = da.quantile(0.999, dim=["ensemble_member", "time"]) # rougly equiv to once every 3 years (at each grid box)
time_q9999 = da.quantile(0.9999, dim=["ensemble_member", "time"]) # roughly equiv to 6 higher values in 12 em, 15 year dataset 

if "rotated_latitude_longitude" in ds.cf.grid_mapping_names:
    projection = cp_model_rotated_pole
elif "transverse_mercator" in ds.cf.grid_mapping_names:
    projection = osgb_crs
elif "latitude_longitude" in ds.cf.grid_mapping_names:
    projection = platecarree
else:
    raise ValueError(f"Unknown grid type: {ds.cf.grid_mapping_names}")


fig, axd = create_map_fig([["mean", "std"], ["q001", "q0001"], ["q999", "q9999"]], projection=projection)
plot_map(time_mean, ax=axd["mean"], style=None, cmap="turbo", title=f"Time mean", add_colorbar=True)
plot_map(time_std , ax=axd["std"], style=None, cmap="turbo", title=f"Time std", add_colorbar=True)
plot_map(time_q001 , ax=axd["q001"], style=None, cmap="turbo", title=f"Time 0.1%ile", add_colorbar=True)
plot_map(time_q0001 , ax=axd["q0001"], style=None, cmap="turbo", title=f"Time 0.01%ile", add_colorbar=True)
plot_map(time_q999 , ax=axd["q999"], style=None, cmap="turbo", title=f"Time 99.9%ile", add_colorbar=True)
plot_map(time_q9999 , ax=axd["q9999"], style=None, cmap="turbo", title=f"Time 99.99%ile", add_colorbar=True)


plt.show()

In [ ]:
client.close()